<a href="https://colab.research.google.com/github/sukrit37/MLPR/blob/main/YOLOv8CustomObjectDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## How to Train YOLOv8 Object Detection on a Custom Dataset

In [25]:
!nvidia-smi

Thu Apr 23 04:26:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [26]:
!pip install ultralytics

## Install YOLOv8

In [27]:
from ultralytics import YOLO
import os
from IPython.display import display, Image
from IPython import display
display.clear_output()
!yolo mode=checks

Traceback (most recent call last):
  File "/usr/local/bin/yolo", line 8, in <module>
    sys.exit(entrypoint())
             ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/__init__.py", line 938, in entrypoint
    raise ValueError(f"Invalid 'mode={mode}'. Valid modes are {list(MODES)}.\n{CLI_HELP_MSG}")
ValueError: Invalid 'mode=checks'. Valid modes are ['predict', 'export', 'val', 'train', 'benchmark', 'track'].

    Arguments received: ['yolo', 'mode=checks']. Ultralytics 'yolo' commands use the following syntax:

        yolo TASK MODE ARGS

        Where   TASK (optional) is one of ['obb', 'pose', 'classify', 'segment', 'detect']
                MODE (required) is one of ['predict', 'export', 'val', 'train', 'benchmark', 'track']
                ARGS (optional) are any number of custom 'arg=value' pairs like 'imgsz=320' that override defaults.
                    See all ARGS at https://docs.ultralytics.com/usage/cfg or with 'yolo cfg'

    1. Train a 

## Inference Example with Pretrained YOLOv8 Model

In [28]:
import os
import shutil
import glob
import yaml
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# --- UPDATE THIS PATH to point to your uploaded CVAT zip file ---
zip_path = "/content/drive/MyDrive/MLPR_Datasets/my_dataset.zip"

# Paths for processing
unzip_dir = "/content/full_cvat_dataset"
subset_dir = "/content/yolo_175_dataset"

# 2. Extract the full dataset
print("Unzipping full dataset...")
!unzip -q "{zip_path}" -d "{unzip_dir}"

# 3. Create the YOLOv8 required directory structure
os.makedirs(f"{subset_dir}/images/train", exist_ok=True)
os.makedirs(f"{subset_dir}/labels/train", exist_ok=True)
# We will use the same 175 images for validation just to get the training pipeline running
os.makedirs(f"{subset_dir}/images/val", exist_ok=True)
os.makedirs(f"{subset_dir}/labels/val", exist_ok=True)

# 4. Find and sort all images to get the first 175
all_images = glob.glob(f"{unzip_dir}/**/*.png", recursive=True) + \
             glob.glob(f"{unzip_dir}/**/*.jpg", recursive=True)
all_images = sorted(all_images)
first_175 = all_images[:175]

# 5. Copy the 175 images and their matching .txt labels
print("Filtering and copying the first 175 frames...")
for img_path in first_175:
    base_name = os.path.basename(img_path)
    name_no_ext = os.path.splitext(base_name)[0]

    # In CVAT YOLO exports, the .txt file is usually in the same folder as the image
    label_path = os.path.join(os.path.dirname(img_path), f"{name_no_ext}.txt")

    # Copy to train (and copy to val for pipeline testing)
    shutil.copy(img_path, f"{subset_dir}/images/train/{base_name}")
    shutil.copy(img_path, f"{subset_dir}/images/val/{base_name}")

    if os.path.exists(label_path):
        shutil.copy(label_path, f"{subset_dir}/labels/train/{name_no_ext}.txt")
        shutil.copy(label_path, f"{subset_dir}/labels/val/{name_no_ext}.txt")
    else:
        # If a frame has no annotations, YOLO needs an empty txt file
        open(f"{subset_dir}/labels/train/{name_no_ext}.txt", 'w').close()
        open(f"{subset_dir}/labels/val/{name_no_ext}.txt", 'w').close()

# 6. Dynamically build data.yaml using CVAT's obj.names file
obj_names_path = glob.glob(f"{unzip_dir}/**/obj.names", recursive=True)

if obj_names_path:
    with open(obj_names_path[0], 'r') as f:
        classes = [line.strip() for line in f.readlines() if line.strip()]
else:
    # Fallback based on your XML snippet just in case
    classes = ['car', 'person', 'license-plate', 'motorbike']

yaml_content = {
    'train': f"{subset_dir}/images/train",
    'val': f"{subset_dir}/images/val",
    'nc': len(classes),
    'names': classes
}

with open(f"{subset_dir}/data.yaml", 'w') as f:
    yaml.dump(yaml_content, f, sort_keys=False)

print(f"Success! Filtered dataset and data.yaml are ready at: {subset_dir}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Unzipping full dataset...
replace /content/full_cvat_dataset/train.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/full_cvat_dataset/obj.names? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/full_cvat_dataset/obj.data? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/full_cvat_dataset/obj_train_data/D1Cam_frame1465350_1765608713087.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/full_cvat_dataset/obj_train_data/D10Cam_frame148515_1765541315993.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/full_cvat_dataset/obj_train_data/D10Cam_frame1906530_1765611920966.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/full_cvat_dataset/obj_train_data/D10Cam_frame2015255_1765616728383.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace /content/full_cvat_dataset/obj_train_data/D10Cam_frame1441270_1765

In [35]:
import glob
all_images = sorted(glob.glob(f"/content/full_cvat_dataset/**/*.png", recursive=True) +
                    glob.glob(f"/content/full_cvat_dataset/**/*.jpg", recursive=True) +
                    glob.glob(f"/content/full_cvat_dataset/**/*.PNG", recursive=True) +
                    glob.glob(f"/content/full_cvat_dataset/**/*.JPG", recursive=True))

print(f"Found {len(all_images)} total images in the extracted folder.")
if len(all_images) == 0:
    print("ERROR: No images found! Please check that your CVAT zip file uploaded correctly and the zip_path is correct.")

Found 0 total images in the extracted folder.
ERROR: No images found! Please check that your CVAT zip file uploaded correctly and the zip_path is correct.


## Train YOLOv8 Model on Custom Dataset

In [31]:
!yolo task=detect mode=train model=yolov8n.pt data=/content/yolo_175_dataset/data.yaml epochs=25 imgsz=640

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_175_dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-4, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patie

In [32]:
!yolo task=detect mode=val model=/content/runs/detect/train/weights/best.pt data={dataset.location}/data.yaml

Traceback (most recent call last):
  File "/usr/local/bin/yolo", line 8, in <module>
    sys.exit(entrypoint())
             ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/__init__.py", line 976, in entrypoint
    model = YOLO(model, task=task)
            ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/models/yolo/model.py", line 76, in __init__
    super().__init__(model=model, task=task, verbose=verbose)
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/model.py", line 144, in __init__
    self._load(model, task=task)
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/model.py", line 283, in _load
    self.model, self.ckpt = load_checkpoint(weights)
                            ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/nn/tasks.py", line 1515, in load_checkpoint
    ckpt, weight = torch_safe_load(weight)  # load ckpt
                   ^^^^^^^^^

In [33]:
!yolo task=detect mode=predict model=/content/runs/detect/train/weights/best.pt conf=0.25 source={dataset.location}/test/images

Traceback (most recent call last):
  File "/usr/local/bin/yolo", line 8, in <module>
    sys.exit(entrypoint())
             ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/cfg/__init__.py", line 976, in entrypoint
    model = YOLO(model, task=task)
            ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/models/yolo/model.py", line 76, in __init__
    super().__init__(model=model, task=task, verbose=verbose)
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/model.py", line 144, in __init__
    self._load(model, task=task)
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/model.py", line 283, in _load
    self.model, self.ckpt = load_checkpoint(weights)
                            ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/nn/tasks.py", line 1515, in load_checkpoint
    ckpt, weight = torch_safe_load(weight)  # load ckpt
                   ^^^^^^^^^

In [34]:
import glob
from IPython.display import Image, display

for image_path in glob.glob(f'/content/runs/detect/predict2/*.jpg')[:5]:
      display(Image(filename=image_path, height=600))
      print("\n")